# 03 — Distill teacher into Qwen2.5-3B (Kaggle T4 x 2)

**Goal:** turn the 14B teacher into a 3B student that runs cheaply at eval / TTT time.

**Why sequence-level distillation (not logit KL):**
- Logit KD requires keeping teacher in memory alongside student — won't fit T4x2 with a 14B teacher.
- Sequence-level (Kim & Rush 2016) is just SFT on teacher-generated outputs. Simple, interpretable, ~equivalent quality for instruction-style tasks.

**Plan:**
1. (Done in notebook 02 or here as a one-shot Colab job) Generate teacher outputs for ~10k augmented prompts → JSONL.
2. Load that JSONL on Kaggle.
3. SFT Qwen2.5-3B-Instruct + LoRA on (prompt, teacher_output) pairs across the 2 T4s with `accelerate`.

Kaggle constraint: **everything inline, no local imports**.


In [ ]:
# Hardware check
import torch, os
print("CUDA:", torch.cuda.is_available(), "| #GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU{i}:", torch.cuda.get_device_name(i),
          round(torch.cuda.get_device_properties(i).total_memory/1e9,1), "GB")


In [ ]:
# Install — Kaggle has most of these; pin for reproducibility
!pip -q install -U transformers==4.46.2 accelerate==1.1.1 bitsandbytes==0.44.1 \
    datasets==3.1.0 peft==0.13.2 trl==0.12.1 sentencepiece


## Step 1 — Build the distillation corpus (run this on Colab H100 first)

Below is the snippet you would run on Colab (where the teacher already lives) to produce `teacher_distill.jsonl`. Then upload that file as a Kaggle dataset.

```python
# === On Colab H100, after notebook 02 ===
# Generates teacher outputs for the augmented training set, writes JSONL.
import json
from tqdm import tqdm

# `model`, `tok`, `augmented`, `format_user`, `SYSTEM` are from notebook 02.
out_path = "/content/teacher_distill.jsonl"
with open(out_path, "w") as f:
    for t in tqdm(augmented[:10000]):
        msgs = [{"role":"system","content":SYSTEM},
                {"role":"user","content":format_user(t)}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tok(text, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inp, max_new_tokens=2048, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        completion = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
        f.write(json.dumps({
            "prompt": format_user(t),
            "teacher_output": completion,
            "ground_truth": format_assistant(t),
        }) + "\n")
```

Once `teacher_distill.jsonl` is uploaded as a Kaggle dataset (e.g. `your-handle/arc-teacher-distill`), continue below.


In [ ]:
# Step 2 — Load the distillation corpus on Kaggle
import json
from pathlib import Path

# Adjust to wherever you uploaded the JSONL
candidate_paths = [
    Path("/kaggle/input/arc-teacher-distill/teacher_distill.jsonl"),
    Path("./teacher_distill.jsonl"),
]
JSONL = next(p for p in candidate_paths if p.exists())
records = [json.loads(l) for l in JSONL.read_text().splitlines()]
print("Distill records:", len(records))
print("Sample keys:", list(records[0]))


In [ ]:
# Step 3 — Format as chat for student SFT
from transformers import AutoTokenizer

STUDENT_ID = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(STUDENT_ID)
if tok.pad_token is None: tok.pad_token = tok.eos_token

SYSTEM = ("You are an expert ARC puzzle solver. Output only the test output grid as digits "
          "separated by spaces, one row per line. No explanations.")

def to_chat(rec):
    msgs = [
        {"role":"system","content":SYSTEM},
        {"role":"user","content":rec["prompt"]},
        {"role":"assistant","content":rec["teacher_output"]},
    ]
    return tok.apply_chat_template(msgs, tokenize=False)

MAX_LEN = 3500
texts = []
for r in records:
    s = to_chat(r)
    if len(tok(s).input_ids) <= MAX_LEN:
        texts.append(s)
print("Distill samples after length filter:", len(texts))


In [ ]:
from datasets import Dataset
ds = Dataset.from_dict({"text": texts}).shuffle(seed=0)

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# device_map="auto" splits across both T4s automatically
model = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir="./student_out",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=20,
    save_steps=1000,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_seq_length=MAX_LEN,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, args=cfg)
trainer.train()

# Save the student LoRA adapter
trainer.model.save_pretrained("./student_out/adapter")
tok.save_pretrained("./student_out/adapter")
print("Saved student adapter.")


## Download the adapter

After training, download `./student_out/adapter` (it's small, ~50MB). You'll plug this into notebook 04 for evaluation + test-time training.
